# SPLED INTELLIGENCE — Free Colab training

This notebook downloads public Binance USDⓈ-M Futures candles, builds seven-strategy labels, trains/calibrates the research model, exports the Android JSON asset, and packages the updated source. It does **not** guarantee profitability or approve real-money trading. Keep the model disabled if the historical gate fails.

In [ ]:
from google.colab import files
import os, zipfile, pathlib, json, subprocess, shutil


## 1. Upload the SPLED source ZIP

In [ ]:
uploaded = files.upload()  # Choose SPLED_INTELLIGENCE_v0.8.0_HYBRID_AI_SOURCE.zip
zip_name = next(iter(uploaded))
root = pathlib.Path('/content/SPLED_INTELLIGENCE')
if root.exists(): shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as z: z.extractall('/content/unpacked')
# The project archive contains a top-level SPLED_INTELLIGENCE folder.
source_root = pathlib.Path('/content/unpacked/SPLED_INTELLIGENCE')
shutil.copytree(source_root, root, dirs_exist_ok=True)
print('Source ready:', root)


## 2. Install free research dependencies

In [ ]:
%cd /content/SPLED_INTELLIGENCE/backend
!python -m pip install -q -r research/requirements.txt


## 3. Download historical market data

Start with liquid contracts. This can take several minutes. Increase the universe only after confirming rate limits and data quality.

In [ ]:
!python research/download_binance_history.py --symbols BTCUSDT,ETHUSDT,BNBUSDT,SOLUSDT,XRPUSDT --days 180 --out research/data_1m.csv


## 4. Build candidate labels

In [ ]:
!python research/label_dataset.py --csv research/data_1m.csv --out research/candidates.csv --horizon 240 --fee-side 0.0005 --slippage-round-trip 0.001


## 5. Train, calibrate, and evaluate

The model exports enabled only if the historical screening gates pass. A pass means **shadow testing only**, never real-money approval.

In [ ]:
!python research/train_walkforward.py --csv research/candidates.csv --out research/artifacts/target_first.joblib --min-oos 200 --android-out ../android/app/src/main/assets/hybrid_model.json


In [ ]:
report = json.loads(pathlib.Path('research/artifacts/target_first.joblib.metrics.json').read_text())
print(json.dumps(report, indent=2))
asset = json.loads(pathlib.Path('../android/app/src/main/assets/hybrid_model.json').read_text())
print('Android shadow model enabled:', asset['enabled_for_shadow'])


## 6. Run tests and package updated source

In [ ]:
%cd /content/SPLED_INTELLIGENCE/backend
!PYTHONPATH=. pytest -q tests research/tests
%cd /content
archive='/content/SPLED_INTELLIGENCE_v0.8.0_trained_shadow_candidate.zip'
if pathlib.Path(archive).exists(): pathlib.Path(archive).unlink()
shutil.make_archive(archive[:-4], 'zip', root_dir='/content', base_dir='SPLED_INTELLIGENCE')
files.download(archive)


## Interpretation

If `Android shadow model enabled` is false, the app will correctly show no AI probability. Do not edit the flag manually. Review holdout metrics, verify Python/Kotlin feature parity, build the APK, and run forward shadow validation (at least 30 days and 200 resolved signals, longer if needed) before considering any live use.